<a href="https://colab.research.google.com/github/jbloombe/Project0/blob/main/We_Love_Sabermetrics_value_projections.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install pybaseball MLB-StatsAPI -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.1/426.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.7/432.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.3 MB/s eta 0:00:00


In [2]:
SAVE_PATH = '/content/drive/MyDrive/WeLoveSabermetrics/'

import os
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"✅ Drive mounted. Saving to: {SAVE_PATH}")

✅ Drive mounted. Saving to: /content/drive/MyDrive/WeLoveSabermetrics/


In [3]:
import pandas as pd

# --- HITTERS ---
hit_url = (
    "https://www.fangraphs.com/api/projections?"
    "type=steamer&stats=bat&pos=all&team=0&players=0&lg=all"
)
hitters = pd.read_json(hit_url)

# --- PITCHERS ---
pit_url = (
    "https://www.fangraphs.com/api/projections?"
    "type=steamer&stats=pit&pos=all&team=0&players=0&lg=all"
)
pitchers = pd.read_json(pit_url)

print(hitters.columns.tolist())   # inspect available columns first
print(pitchers.columns.tolist())


['Team', 'ShortName', 'G', 'AB', 'PA', 'H', '1B', '2B', '3B', 'HR', 'R', 'RBI', 'BB', 'IBB', 'SO', 'HBP', 'SF', 'SH', 'GDP', 'SB', 'CS', 'AVG', 'OBP', 'SLG', 'OPS', 'wOBA', 'BB%', 'K%', 'BB/K', 'ISO', 'Spd', 'BABIP', 'UBR', 'GDPRuns', 'wRC', 'wRAA', 'UZR', 'wBsR', 'BaseRunning', 'WAR', 'Off', 'Def', 'wRC+', 'FPTS', 'FPTS_G', 'SPTS', 'SPTS_G', 'woba_sd', 'truetalent_sd', 'woba_sd_book', 'woba_se', 'total_se', 'q10', 'q20', 'q30', 'q40', 'q50', 'q60', 'q70', 'q80', 'q90', 'tt_q10', 'tt_q20', 'tt_q30', 'tt_q40', 'tt_q50', 'tt_q60', 'tt_q70', 'tt_q80', 'tt_q90', 'ADP', 'Pos', 'minpos', 'UPURL', 'teamid', 'League', 'PlayerName', 'xMLBAMID', 'playerids', 'playerid']
['Team', 'ShortName', 'W', 'L', 'GS', 'G', 'SV', 'HLD', 'BS', 'IP', 'TBF', 'H', 'R', 'ER', 'HR', 'SO', 'BB', 'IBB', 'HBP', 'ERA', 'WHIP', 'K/9', 'BB/9', 'K/BB', 'HR/9', 'K%', 'BB%', 'K-BB%', 'GB%', 'AVG', 'BABIP', 'LOB%', 'FIP', 'FPTS', 'FPTS_IP', 'SPTS', 'SPTS_IP', 'WAR', 'RA9-WAR', 'QS', 'ra_talent_sd', 'chance_ra_se', 'total_r

# Scoring

In [4]:
# ── HITTER SCORING (vectorized — fixes NaN bug) ──────────────────────────────
h = hitters.fillna(0)
hitters['fantasy_pts'] = (
    h['1B']  * 1.0   +
    h['2B']  * 2.0   +
    h['3B']  * 3.0   +
    h['HR']  * 4.0   +
    h['RBI'] * 1.0   +
    h['R']   * 1.0   +
    h['SB']  * 2.0   +
    h['BB']  * 1.0   +
    h['HBP'] * 0.5   +
    h['SO']  * -0.5  +
    h['CS']  * -0.5  +
    h['GDP'] * -0.5
)

# ── PITCHER SCORING (vectorized — QS and BS direct from data) ────────────────
p = pitchers.fillna(0)
pitchers['fantasy_pts'] = (
    p['IP']  * 1.25   +
    p['ER']  * -1.5   +
    p['H']   * -0.625 +
    p['BB']  * -0.625 +
    p['SO']  * 1.125  +
    p['W']   * 3.0    +
    p['L']   * -2.75  +
    p['SV']  * 4.0    +
    p['HLD'] * 2.5    +
    p['BS']  * -2.75  +
    p['QS']  * 5.0    +
    p['HBP'] * -0.5
)

# ── RANKED OUTPUTS ───────────────────────────────────────────────────────────
hit_ranked = hitters[['PlayerName', 'Team', 'Pos', 'PA', 'HR', 'SB', 'BB', 'SO', 'fantasy_pts']]\
    .sort_values('fantasy_pts', ascending=False)\
    .reset_index(drop=True)
hit_ranked.index += 1

pit_ranked = pitchers[['PlayerName', 'Team', 'GS', 'G', 'IP', 'ERA', 'SO', 'SV', 'HLD', 'QS', 'fantasy_pts']]\
    .sort_values('fantasy_pts', ascending=False)\
    .reset_index(drop=True)
pit_ranked.index += 1

print("=== TOP 30 HITTERS ===")
print(hit_ranked.head(30).to_string())

print("\n=== TOP 30 PITCHERS ===")
print(pit_ranked.head(30).to_string())

# ── SAVE ─────────────────────────────────────────────────────────────────────
hit_ranked.to_csv('hitter_rankings_2026.csv', index_label='Rank')
pit_ranked.to_csv('pitcher_rankings_2026.csv', index_label='Rank')

print("\n✅ CSVs saved: hitter_rankings_2026.csv & pitcher_rankings_2026.csv")


=== TOP 30 HITTERS ===
               PlayerName Team       Pos       PA       HR       SB        BB        SO  fantasy_pts
1           Shohei Ohtani  LAD  0.918605  658.176  43.5958  22.2879   91.4865  160.5680    594.59485
2             Aaron Judge  NYY  0.370861  633.319  42.2973   8.8032  112.2240  156.1420    561.17035
3               Juan Soto  NYM  0.018750  621.171  34.2478  19.6565  117.1530  110.5420    556.59400
4        Ronald Acuña Jr.  ATL  0.042105  676.219  30.8111  24.1558   90.7485  135.4620    552.52770
5          Bobby Witt Jr.  KCR  0.025316  633.655  27.9706  31.2113   47.0172  105.8900    540.02370
6   Vladimir Guerrero Jr.  TOR  0.152866  636.639  32.1725   4.7888   73.6591   87.5248    526.88565
7        Gunnar Henderson  BAL  0.052288  665.535  28.1302  23.5706   70.8129  134.2250    519.62460
8            José Ramírez  CLE  0.162500  603.880  26.8873  29.7851   58.2744   72.4692    514.36555
9          Corbin Carroll  ARI  0.020833  604.763  26.4819  32.2307 

# Master Board

In [53]:
# Tag each group so we know the player type after merging
hitters['player_type']  = 'Hitter'
pitchers['player_type'] = 'Pitcher'

# Pitchers don't have a Pos column like hitters do — assign SP or RP
# based on whether they have 5+ projected starts
pitchers['Pos'] = pitchers.apply(
    lambda r: 'SP' if r['GS'] >= 5 else 'RP', axis=1
)

# Columns we want in the combined board
shared = ['PlayerName', 'Team', 'Pos', 'player_type', 'fantasy_pts', 'ADP', 'xMLBAMID']

# Stack hitters and pitchers into one dataframe, sorted by our custom score
master = pd.concat([
    hitters[shared],
    pitchers[shared]
]).sort_values('fantasy_pts', ascending=False).reset_index(drop=True)
master.index += 1  # start rank at 1 instead of 0

# Positive value = market is sleeping on this player relative to your scoring
master['value_vs_adp'] = master['ADP'] - master.index

print(master.head(30).to_string())
master.to_csv('master_draft_board_2026.csv', index_label='Overall_Rank')
print("\n✅ master_draft_board_2026.csv saved")


               PlayerName Team       Pos player_type  fantasy_pts         ADP  xMLBAMID  value_vs_adp
1           Shohei Ohtani  LAD  0.918605      Hitter    594.59485    1.270000  660271.0      0.270000
2             Aaron Judge  NYY  0.370861      Hitter    561.17035    1.880000  592450.0     -0.120000
3               Juan Soto  NYM   0.01875      Hitter    556.59400    4.590000  665742.0      1.590000
4        Ronald Acuña Jr.  ATL  0.042105      Hitter    552.52770    5.710000  660670.0      1.710000
5          Bobby Witt Jr.  KCR  0.025316      Hitter    540.02370    2.990000  677951.0     -2.010000
6   Vladimir Guerrero Jr.  TOR  0.152866      Hitter    526.88565   18.059999  665489.0     12.059999
7        Gunnar Henderson  BAL  0.052288      Hitter    519.62460   12.100000  683002.0      5.100000
8            José Ramírez  CLE    0.1625      Hitter    514.36555    5.820000  608070.0     -2.180000
9          Corbin Carroll  ARI  0.020833      Hitter    504.02525   15.060000  682

In [54]:
# Hitter output
hit_ranked = hitters[['PlayerName', 'Team', 'Pos', 'PA', 'HR', 'SB', 'fantasy_pts']]\
    .sort_values('fantasy_pts', ascending=False)\
    .reset_index(drop=True)
hit_ranked.index += 1

# Pitcher output
pit_ranked = pitchers[['PlayerName', 'Team', 'GS', 'IP', 'ERA', 'SO', 'SV', 'fantasy_pts']]\
    .sort_values('fantasy_pts', ascending=False)\
    .reset_index(drop=True)
pit_ranked.index += 1

print("=== TOP 30 HITTERS ===")
print(hit_ranked.head(30).to_string())

print("\n=== TOP 30 PITCHERS ===")
print(pit_ranked.head(30).to_string())

hit_ranked.to_csv('hitter_rankings_2026.csv', index_label='Rank')
pit_ranked.to_csv('pitcher_rankings_2026.csv', index_label='Rank')


=== TOP 30 HITTERS ===
               PlayerName Team       Pos       PA       HR       SB  fantasy_pts
1           Shohei Ohtani  LAD  0.918605  658.176  43.5958  22.2879    594.59485
2             Aaron Judge  NYY  0.370861  633.319  42.2973   8.8032    561.17035
3               Juan Soto  NYM  0.018750  621.171  34.2478  19.6565    556.59400
4        Ronald Acuña Jr.  ATL  0.042105  676.219  30.8111  24.1558    552.52770
5          Bobby Witt Jr.  KCR  0.025316  633.655  27.9706  31.2113    540.02370
6   Vladimir Guerrero Jr.  TOR  0.152866  636.639  32.1725   4.7888    526.88565
7        Gunnar Henderson  BAL  0.052288  665.535  28.1302  23.5706    519.62460
8            José Ramírez  CLE  0.162500  603.880  26.8873  29.7851    514.36555
9          Corbin Carroll  ARI  0.020833  604.763  26.4819  32.2307    504.02525
10        Freddie Freeman  LAD  0.000000  658.412  24.1366   7.2019    478.59065
11        Elly De La Cruz  CIN  0.024691  616.808  23.3673  37.5639    475.62185
12   

In [55]:
# Tag each group
hitters['player_type']  = 'Hitter'
pitchers['player_type'] = 'Pitcher'

# Align shared columns
shared = ['PlayerName', 'Team', 'Pos', 'player_type', 'fantasy_pts', 'ADP', 'xMLBAMID']

# Pitchers don't have Pos in the same format — fill it
pitchers['Pos'] = pitchers.apply(
    lambda r: 'SP' if r['GS'] >= 5 else 'RP', axis=1
)

master = pd.concat([
    hitters[shared],
    pitchers[shared]
]).sort_values('fantasy_pts', ascending=False).reset_index(drop=True)
master.index += 1

print(master.head(50).to_string())
master.to_csv('master_draft_board_2026.csv', index_label='Overall_Rank')


               PlayerName Team       Pos player_type  fantasy_pts         ADP  xMLBAMID
1           Shohei Ohtani  LAD  0.918605      Hitter   594.594850    1.270000  660271.0
2             Aaron Judge  NYY  0.370861      Hitter   561.170350    1.880000  592450.0
3               Juan Soto  NYM   0.01875      Hitter   556.594000    4.590000  665742.0
4        Ronald Acuña Jr.  ATL  0.042105      Hitter   552.527700    5.710000  660670.0
5          Bobby Witt Jr.  KCR  0.025316      Hitter   540.023700    2.990000  677951.0
6   Vladimir Guerrero Jr.  TOR  0.152866      Hitter   526.885650   18.059999  665489.0
7        Gunnar Henderson  BAL  0.052288      Hitter   519.624600   12.100000  683002.0
8            José Ramírez  CLE    0.1625      Hitter   514.365550    5.820000  608070.0
9          Corbin Carroll  ARI  0.020833      Hitter   504.025250   15.060000  682998.0
10        Freddie Freeman  LAD       0.0      Hitter   478.590650   67.129997  518692.0
11        Elly De La Cruz  CIN  

In [56]:
master['value_vs_adp'] = master['ADP'] - master.index  # positive = undervalued

undervalued = master[master['value_vs_adp'] > 10]\
    .sort_values('value_vs_adp', ascending=False)

print("=== BIGGEST VALUES vs ADP ===")
print(undervalued.head(20).to_string())


=== BIGGEST VALUES vs ADP ===
            PlayerName Team       Pos player_type  fantasy_pts    ADP  xMLBAMID  value_vs_adp
55   Munetaka Murakami  CHW       0.0      Hitter    397.45230  999.0  808959.0         944.0
93      Kazuma Okamoto  TOR       0.0      Hitter    341.77095  999.0  672960.0         906.0
108    Kevin McGonigle  DET       0.0      Hitter    322.11730  999.0  805808.0         891.0
144      JJ Wetherholt  STL       0.0      Hitter    300.08125  999.0  802139.0         855.0
200     Chase DeLauter  CLE       0.0      Hitter    257.76705  999.0  800050.0         799.0
208      Wenceel Pérez  DET  0.008772      Hitter    252.94675  999.0  672761.0         791.0
236       Joc Pederson  TEX  0.942529      Hitter    238.70125  999.0  592626.0         763.0
244     Ke'Bryan Hayes  CIN       0.0      Hitter    233.44090  999.0  663647.0         755.0
247     Brayan Rocchio  CLE       0.0      Hitter    232.51700  999.0  677587.0         752.0
264     Konnor Griffin  PIT   

# Adding Savant Percentiles

In [46]:
from pybaseball import statcast_batter_percentile_ranks, statcast_pitcher_percentile_ranks

In [52]:
print('xMLBAMID' in master.columns)
print(master.columns.tolist())


False
['PlayerName', 'Team', 'Pos', 'player_type', 'fantasy_pts', 'ADP', 'value_vs_adp']


In [57]:
# ── PULL FRESH — raw variables never get overwritten ─────────────────────────
_batter_raw  = statcast_batter_percentile_ranks(2025)
_pitcher_raw = statcast_pitcher_percentile_ranks(2025)

print("BATTER COLUMNS:", _batter_raw.columns.tolist())  # run once to confirm, then remove
print("PITCHER COLUMNS:", _pitcher_raw.columns.tolist())

# ── HITTER SAVANT SCORE ───────────────────────────────────────────────────────
# These are raw Statcast values, not percentiles — normalize to 0-100 first
batter_cols = ['xba', 'xslg', 'brl_percent', 'hard_hit_percent', 'sprint_speed']

def pct_rank(series):
    return series.rank(pct=True) * 100

_batter_raw['savant_score'] = (
    pct_rank(_batter_raw['xba']) +
    pct_rank(_batter_raw['xslg']) +
    pct_rank(_batter_raw['brl_percent']) +
    pct_rank(_batter_raw['hard_hit_percent']) +
    pct_rank(_batter_raw['sprint_speed'])
).div(5)

_batter_raw['savant_mult']  = 1.0 + (_batter_raw['savant_score'] - 50) / 250
batter_savant = _batter_raw[['player_id', 'savant_mult']].rename(columns={'player_id': 'xMLBAMID'})

# ── PITCHER SAVANT SCORE ──────────────────────────────────────────────────────
# xera is "lower is better" so invert it with negative rank
_pitcher_raw['savant_score'] = (
    pct_rank(-_pitcher_raw['xera']) +       # inverted — lower xERA = better
    pct_rank(_pitcher_raw['k_percent']) +
    pct_rank(_pitcher_raw['whiff_percent']) +
    pct_rank(-_pitcher_raw['bb_percent'])    # inverted — lower BB% = better
).div(4)

_pitcher_raw['savant_mult']  = 1.0 + (_pitcher_raw['savant_score'] - 50) / 250
pitcher_savant = _pitcher_raw[['player_id', 'savant_mult']].rename(columns={'player_id': 'xMLBAMID'})

# ── MERGE INTO MASTER ─────────────────────────────────────────────────────────
savant_all = pd.concat([batter_savant, pitcher_savant]).drop_duplicates('xMLBAMID')
master = master.merge(savant_all, on='xMLBAMID', how='left')
master['savant_mult'] = master['savant_mult'].fillna(1.0)

print(f"✅ Savant merged — {master['savant_mult'].gt(1).sum()} players above average")
print(master[['PlayerName', 'savant_mult']].head(15).to_string())


BATTER COLUMNS: ['player_name', 'player_id', 'year', 'xwoba', 'xba', 'xslg', 'xiso', 'xobp', 'brl', 'brl_percent', 'exit_velocity', 'max_ev', 'hard_hit_percent', 'k_percent', 'bb_percent', 'whiff_percent', 'chase_percent', 'arm_strength', 'sprint_speed', 'oaa', 'bat_speed', 'squared_up_rate', 'swing_length']
PITCHER COLUMNS: ['player_name', 'player_id', 'year', 'xwoba', 'xba', 'xslg', 'xiso', 'xobp', 'brl', 'brl_percent', 'exit_velocity', 'max_ev', 'hard_hit_percent', 'k_percent', 'bb_percent', 'whiff_percent', 'chase_percent', 'arm_strength', 'xera', 'fb_velocity', 'fb_spin', 'curve_spin']
✅ Savant merged — 305 players above average
               PlayerName  savant_mult
0           Shohei Ohtani     1.162318
1             Aaron Judge     1.152053
2               Juan Soto     1.122056
3        Ronald Acuña Jr.     1.136221
4          Bobby Witt Jr.     1.156937
5   Vladimir Guerrero Jr.     1.115383
6        Gunnar Henderson     1.081536
7            José Ramírez     1.036455
8      

# Importing Prospect and Dynasty Rankings

In [9]:
import gspread
from google.colab import auth
from google.auth import default

# This is the correct auth method for Colab — triggers a one-time popup
auth.authenticate_user()

creds, _ = default(scopes=[
    'https://spreadsheets.google.com/feeds',
    'https://www.googleapis.com/auth/drive'
])

gc = gspread.authorize(creds)
print("✅ Google Sheets authenticated")


✅ Google Sheets authenticated


In [10]:
# ── PROSPECT DATA — load from full CSV instead of Google Sheet ────────────────
prospect_data = pd.read_csv('/content/drive/MyDrive/2026 Top Prospects Fangraphs - Sheet1.csv')

# Clean FV — strips "40+" → 40, "45+" → 45, etc.
prospect_data['FV'] = prospect_data['FV'].astype(str).str.replace('+', '', regex=False)
prospect_data['FV'] = pd.to_numeric(prospect_data['FV'], errors='coerce').fillna(40).astype(int)

# Add 'Name_clean' column to prospect_data
prospect_data['Name_clean'] = prospect_data['Name'].str.strip().str.lower()

# Load top 500 rankings sheet
top500_sh   = gc.open("Fantrax Top 500 Rankings")  # update to exact sheet name
top500_data = pd.DataFrame(top500_sh.sheet1.get_all_records())
top500_data['Name_clean'] = top500_data['Player'].str.strip().str.lower() # Changed 'Name' to 'Player'

print("✅ Prospect sheet loaded:", len(prospect_data), "players")
print("✅ Top 500 sheet loaded:", len(top500_data), "players")

✅ Prospect sheet loaded: 200 players
✅ Top 500 sheet loaded: 503 players


In [11]:
import unicodedata

def clean_name(name):
    # Lowercase, strip whitespace, remove accents
    name = str(name).strip().lower()
    return ''.join(
        c for c in unicodedata.normalize('NFD', name)
        if unicodedata.category(c) != 'Mn'
    )

# ── PROSPECT DATA ─────────────────────────────────────────────────────────────
prospect_data['Name_clean'] = prospect_data['Name'].apply(clean_name)

# ── TOP 500 ───────────────────────────────────────────────────────────────────
top500_data['Name_clean'] = top500_data['Player'].apply(clean_name)


In [12]:
def get_player_rank(player_name, default_rank=999):
    """
    Look up a player's rank from your top 500 sheet.
    Returns 999 if not found (unranked).
    """
    match = top500_data[top500_data['Name_clean'] == player_name.strip().lower()]
    if not match.empty:
        return int(match.iloc[0]['Rank'])  # update 'Rank' to match your column name
    return default_rank


# League Teams

In [13]:
# Load the Fantrax league export you uploaded to Colab
fantrax = pd.read_csv('/content/drive/MyDrive/Fantrax-Players-We-Love-Sabermetrics.csv')

# Convert numeric columns — errors='coerce' turns any bad values into NaN
fantrax['FPts'] = pd.to_numeric(fantrax['FPts'], errors='coerce').fillna(0)
fantrax['Age']  = pd.to_numeric(fantrax['Age'],  errors='coerce')
fantrax['ADP']  = pd.to_numeric(fantrax['ADP'],  errors='coerce')

# Map Fantrax's short team abbreviations to full team names
team_map = {
    'SEAMHEAD' : 'The Seamhead Express',
    'HoF'      : 'Hoos on First',
    'WOOD'     : 'morning WOOD',
    'SHW'      : 'The Boston Red Shawx',
    'MBC'      : "Madoff's Biggest Client",
    'ZL'       : 'New Team 6',
    'AT'       : 'Ashcrafts or Tatis',
    'MB'       : 'Markyball',
    'Luka'     : 'Dick Pic Mick Redemption Rally',
    'DEAL'     : 'Wine and Dynasty',
    'LGO'      : "Spinmaster Wizards",
    'HCGA'     : "Harry Caray's Green Apples",
    'Nunez'    : 'Baltimore KnOrioles',
    '5B DSC'   : '$5B Dollar Shave Club',

}

fantrax['FantasyTeam'] = fantrax['Status'].map(team_map).fillna('FA')

# Keeper Template

In [14]:
'''

# Generate a blank keeper template for all 14 teams
teams = [
    "Harry Caray's Green Apples", "Markyball", "morning WOOD",
    "Wine and Dynasty", "Madoff's Biggest Client", "Spinmaster Wizards",
    "The Boston Red Shawx", "Hoos on First", "The Seamhead Express",
    "Ashcrafts or Tatis", "Dick Pic Mick Redemption Rally",
    "New Team 6", "$5B Dollar Shave Club", "Baltimore KnOrioles"
]

# Create empty template
keeper_template = pd.DataFrame({
    'PlayerName': [''] * 22 * 14,
    'FantasyTeam': [t for t in teams for _ in range(22)],
    'IsProspect':  [False] * 22 * 14   # flag minor leaguers
})

keeper_template.to_csv('keeper_template.csv', index=False)
print("✅ keeper_template.csv saved — fill this in and re-upload")

'''


'\n\n# Generate a blank keeper template for all 14 teams\nteams = [\n    "Harry Caray\'s Green Apples", "Markyball", "morning WOOD",\n    "Wine and Dynasty", "Madoff\'s Biggest Client", "Spinmaster Wizards",\n    "The Boston Red Shawx", "Hoos on First", "The Seamhead Express",\n    "Ashcrafts or Tatis", "Dick Pic Mick Redemption Rally",\n    "New Team 6", "$5B Dollar Shave Club", "Baltimore KnOrioles"\n]\n\n# Create empty template\nkeeper_template = pd.DataFrame({\n    \'PlayerName\': [\'\'] * 22 * 14,\n    \'FantasyTeam\': [t for t in teams for _ in range(22)],\n    \'IsProspect\':  [False] * 22 * 14   # flag minor leaguers\n})\n\nkeeper_template.to_csv(\'keeper_template.csv\', index=False)\nprint("✅ keeper_template.csv saved — fill this in and re-upload")\n\n'

In [21]:
# Age-based multiplier — younger players are worth more in dynasty
# because they have more seasons of value ahead of them
def dynasty_age_multiplier(age):
    if age <= 22:   return 1.35
    elif age <= 24: return 1.25
    elif age <= 26: return 1.15
    elif age <= 28: return 1.05
    elif age <= 30: return 1.00
    elif age <= 32: return 0.88
    elif age <= 34: return 0.75
    else:           return 0.60

# Pull only your roster from the full Fantrax data
my_team   = 'The Seamhead Express'
my_roster = fantrax[fantrax['FantasyTeam'] == my_team].copy()

# Calculate dynasty score = Fantrax points × age multiplier
my_roster['age_mult']      = my_roster['Age'].apply(dynasty_age_multiplier)
my_roster['dynasty_score'] = my_roster['FPts'] * my_roster['age_mult']
my_roster['is_prospect']   = my_roster['FPts'] == 0

# FV-based scores for pure prospects (FPts = 0)
# Adjust FV grades here based on your own scouting opinion
fv_to_score = {80: 600, 70: 500, 60: 400, 55: 320, 50: 250, 45: 170, 40: 100}
prospect_fv = {
    'Max Clark'          : 60,   # #7 overall
    'George Lombard Jr.' : 50,   # #49 overall
    'Travis Sykora'      : 50,   # #109 overall
    'Druw Jones'         : 45,   # unranked
    'Bo Davidson'        : 45,   # unranked
}

# Override dynasty_score for prospects using FV × age multiplier instead of FPts
my_roster['dynasty_score'] = my_roster.apply(
    lambda r: fv_to_score.get(prospect_fv.get(r['Player'], 40), 100) * r['age_mult']
    if r['is_prospect'] else r['dynasty_score'],
    axis=1
)

# Sort best to worst and rank from 1
my_roster = my_roster.sort_values('dynasty_score', ascending=False).reset_index(drop=True)
my_roster.index += 1

# Top 22 are recommended keepers
my_roster['KEEP'] = my_roster.index <= 22

print(my_roster[['Player', 'Position', 'Age', 'FPts', 'dynasty_score', 'is_prospect', 'KEEP']].to_string())
my_roster.to_csv('/content/drive/MyDrive/WeLoveSabermetrics/seamhead_keeper_decisions.csv', index_label='Rank')
print("\n✅ Saved to Drive")


                Player Position  Age    FPts  dynasty_score  is_prospect   KEEP
1          Aaron Judge       OF   33  677.00       507.7500        False   True
2           Matt Olson       1B   31  506.00       445.2800        False   True
3             Ben Rice     C,1B   27  394.50       414.2250        False   True
4          Sal Stewart       1B   22  305.50       412.4250        False   True
5      Shea Langeliers        C   28  381.50       400.5750        False   True
6             Ian Happ       OF   31  436.50       384.1200        False   True
7        Anthony Volpe       SS   24  285.50       356.8750        False   True
8       Bryan Reynolds       OF   31  402.50       354.2000        False   True
9          Chase Burns       SP   23  276.50       345.6250        False   True
10     Brendan Donovan       2B   29  340.00       340.0000        False   True
11  George Lombard Jr.       SS   20    0.00       337.5000         True   True
12       Travis Sykora       SP   21    

In [62]:
## Value scores
# ── FV GRADE → BASE SCORE ─────────────────────────────────────────────────────
fv_to_score = {80: 500, 70: 400, 65: 350, 60: 300, 55: 240, 50: 180, 45: 120, 40: 80}

# ── PROSPECT FV LOOKUP (was missing from notebook) ────────────────────────────
def get_prospect_fv(player_name, default_fv=40):
    match = prospect_data[prospect_data['Name_clean'] == clean_name(player_name)]
    if not match.empty:
        return int(match.iloc[0]['FV'])
    return default_fv


# ── TOP 500 RANK LOOKUP (fixes column name — uses 'Points' not 'Rank') ────────
def get_player_rank(player_name, default_rank=999):
    match = top500_data[top500_data['Name_clean'] == clean_name(player_name)]
    if not match.empty:
        return int(match.iloc[0]['Points'])
    return default_rank

# ── RANK BONUS — rewards rank 1 vs rank 100 within same FV tier ──────────────
def rank_bonus(rank, max_rank=110):
    if rank >= 999:
        return 0
    return round((1 - (rank - 1) / max_rank) * 100)

# ── Players to evaluate using FV instead of FPts (partial MLB stats skewing) ──
# Add any player here whose FPts underrepresent their true dynasty value
fv_override = {'Max Clark', 'Chase Dollander', 'Logan Henderson',
               'Owen Caissie', 'Sal Stewart'}

# ── UNIFIED VALUE SCORE ───────────────────────────────────────────────────────
def calc_value_score(row):
    age_mult      = dynasty_age_multiplier(row['Age'])
    pure_prospect = row['FPts'] == 0
    has_real_stats = row['FPts'] > 150
    use_fv        = (pure_prospect or row['Player'] in fv_override) and not has_real_stats

    if use_fv:
        fv    = get_prospect_fv(row['Player'])
        base  = fv_to_score.get(fv, 80)
        p_row = prospect_data[prospect_data['Name_clean'] == clean_name(row['Player'])]
        rk    = int(p_row.iloc[0]['Top 100']) if not p_row.empty else 999
        bonus = rank_bonus(rk)
        return (base + bonus) * age_mult          # ← prospects don't get savant_mult
    else:
        projected = row.get('fantasy_pts', 0)
        projected = 0 if pd.isna(projected) else projected
        current   = row.get('FPts', 0) or 0
        blended   = (0.6 * projected) + (0.4 * current)
        savant    = row.get('savant_mult', 1.0)
        savant    = 1.0 if pd.isna(savant) else savant
        return blended * age_mult * savant


# ── MERGE PROJECTED STATS INTO ROSTER ────────────────────────────────────────
# Only merge if fantasy_pts column doesn't already exist from a previous run
if 'fantasy_pts' not in my_roster.columns:
    my_roster = my_roster.merge(
        master[['PlayerName', 'fantasy_pts']].rename(columns={'PlayerName': 'Player'}),
        on='Player',
        how='left'
    )

# ── MERGE PROJECTED STATS INTO ROSTER ────────────────────────────────────────
if 'fantasy_pts' not in my_roster.columns:
    my_roster = my_roster.merge(
        master[['PlayerName', 'fantasy_pts']].rename(columns={'PlayerName': 'Player'}),
        on='Player', how='left'
    )

# ── MERGE SAVANT MULTIPLIER INTO ROSTER ──────────────────────────────────────
if 'savant_mult' not in my_roster.columns:
    my_roster = my_roster.merge(
        master[['PlayerName', 'savant_mult']].rename(columns={'PlayerName': 'Player'}),
        on='Player', how='left'
    )
    my_roster['savant_mult'] = my_roster['savant_mult'].fillna(1.0)

# ── CALCULATE VALUE SCORES ────────────────────────────────────────────────────
my_roster['value_score'] = my_roster.apply(calc_value_score, axis=1)
my_roster = my_roster.sort_values('value_score', ascending=False).reset_index(drop=True)
my_roster.index += 1
my_roster['KEEP'] = my_roster.index <= 22


print(my_roster[['Player', 'Position', 'Age', 'FPts', 'fantasy_pts', 'value_score', 'KEEP']].to_string())
my_roster.to_csv('/content/drive/MyDrive/WeLoveSabermetrics/seamhead_keeper_decisions.csv', index_label='Rank')
print("\n✅ Saved to Drive")

                Player Position  Age    FPts  fantasy_pts  value_score   KEEP
1            Max Clark       OF   21   47.50     1.290050   533.250000   True
2          Aaron Judge       OF   33  677.00   561.170350   524.906081   True
3           Matt Olson       1B   31  506.00   455.748800   444.720102   True
4             Ben Rice     C,1B   27  394.50   331.173500   429.126163   True
5      Shea Langeliers        C   28  381.50   347.892200   404.966055   True
6          Sal Stewart       1B   22  305.50   280.209950   391.940060   True
7             Ian Happ       OF   31  436.50   399.912700   378.884517   True
8       Bryan Reynolds       OF   31  402.50   366.016650   354.382404   True
9      Brendan Donovan       2B   29  340.00   343.101450   348.577163   True
10        Matt Chapman       3B   32  381.50   379.682700   345.302069   True
11       Anthony Volpe       SS   24  285.50   266.311650   338.171548   True
12       Trent Grisham       OF   29  303.00   297.790500   318.

In [69]:
requested_players = ['Matt Olson']

for player_name in requested_players:
    player_data = my_roster[my_roster['Player'] == player_name]
    if not player_data.empty:
        print(f"Player: {player_name}, Value Score: {player_data['value_score'].iloc[0]:.2f}")
    else:
        print(f"Player '{player_name}' not found in 'my_roster'.")


Player: Matt Olson, Value Score: 444.72


In [65]:
player_names_to_evaluate = ['Jacob Wilson', 'Nick Lodolo', 'Taylor Ward']

for player_name in player_names_to_evaluate:
    # Get player data from fantrax (for age and FPts)
    player_fantrax_data = fantrax[fantrax['Player'] == player_name]

    # Get player data from master (for fantasy_pts and player_type)
    player_master_data = master[master['PlayerName'] == player_name]

    if not player_fantrax_data.empty and not player_master_data.empty:
        # Create a dictionary representing a row for calc_value_score
        player_row = {
            'Player': player_name,
            'Age': player_fantrax_data['Age'].iloc[0],
            'FPts': player_fantrax_data['FPts'].iloc[0],
            'fantasy_pts': player_master_data['fantasy_pts'].iloc[0]
        }

        # Convert to a Series to simulate a DataFrame row for apply
        player_series = pd.Series(player_row)

        # Calculate value_score
        value_score = calc_value_score(player_series)
        print(f"Player: {player_name}, Value Score: {value_score:.2f}")
    else:
        print(f"Player '{player_name}' not found in the league-wide player pool data.")

Player: Jacob Wilson, Value Score: 437.05
Player: Nick Lodolo, Value Score: 239.79
Player: Taylor Ward, Value Score: 357.00


# Player Breakdowns

In [73]:
def player_breakdown(player_name):
    player_fantrax_data = fantrax[fantrax['Player'] == player_name]
    player_master_data  = master[master['PlayerName'] == player_name]

    if player_fantrax_data.empty or player_master_data.empty:
        print(f"❌ '{player_name}' not found — check spelling or try a partial match:")
        print(fantrax[fantrax['Player'].str.contains(player_name, case=False)]['Player'].tolist())
        return

    age        = player_fantrax_data['Age'].iloc[0]
    fpts       = player_fantrax_data['FPts'].iloc[0]
    fantasy_pts = player_master_data['fantasy_pts'].iloc[0]
    savant     = player_master_data['savant_mult'].iloc[0]
    age_mult   = dynasty_age_multiplier(age)
    blended    = (0.6 * fantasy_pts) + (0.4 * fpts)
    final      = blended * age_mult * savant
    team       = player_fantrax_data['FantasyTeam'].iloc[0]
    adp        = player_master_data['ADP'].iloc[0]

    print(f"{'─'*40}")
    print(f"  {player_name} ({team})")
    print(f"{'─'*40}")
    print(f"  Age:         {age}  →  age_mult: {age_mult:.2f}x")
    print(f"  FPts:        {fpts:.2f}  (2025 actual)")
    print(f"  fantasy_pts: {fantasy_pts:.2f}  (2026 projected)")
    print(f"  savant_mult: {savant:.4f}")
    print(f"  blended:     {blended:.2f}")
    print(f"  ADP:         {adp:.1f}")
    print(f"{'─'*40}")
    print(f"  VALUE SCORE: {final:.2f}")
    print(f"{'─'*40}")

# ── CHANGE THIS TO ANY PLAYER ─────────────────────────────────────────────────
player_breakdown('Jacob Wilson')
player_breakdown('Matt Olson')
player_breakdown('Nick Lodolo')
player_breakdown('Taylor Ward')


────────────────────────────────────────
  Jacob Wilson (Hoos on First)
────────────────────────────────────────
  Age:         23  →  age_mult: 1.25x
  FPts:        385.50  (2025 actual)
  fantasy_pts: 325.74  (2026 projected)
  savant_mult: 0.9250
  blended:     349.64
  ADP:         186.4
────────────────────────────────────────
  VALUE SCORE: 404.28
────────────────────────────────────────
────────────────────────────────────────
  Matt Olson (The Seamhead Express)
────────────────────────────────────────
  Age:         31  →  age_mult: 0.88x
  FPts:        506.00  (2025 actual)
  fantasy_pts: 455.75  (2026 projected)
  savant_mult: 1.0620
  blended:     475.85
  ADP:         49.5
────────────────────────────────────────
  VALUE SCORE: 444.72
────────────────────────────────────────
────────────────────────────────────────
  Nick Lodolo (New Team 6)
────────────────────────────────────────
  Age:         28  →  age_mult: 1.05x
  FPts:        250.12  (2025 actual)
  fantasy_pts: 213

In [74]:
# ── KEEPERS ───────────────────────────────────────────────────
print("=== KEEPERS (Top 22) ===")
for player in my_roster[my_roster['KEEP'] == True]['Player']:
    player_breakdown(player)

# ── ON THE BUBBLE ─────────────────────────────────────────────
print("\n=== ON THE BUBBLE (23–27) ===")
for player in my_roster[my_roster['KEEP'] == False]['Player'].head(5):
    player_breakdown(player)


=== KEEPERS (Top 22) ===
────────────────────────────────────────
  Max Clark (The Seamhead Express)
────────────────────────────────────────
  Age:         21  →  age_mult: 1.35x
  FPts:        47.50  (2025 actual)
  fantasy_pts: 1.29  (2026 projected)
  savant_mult: 1.0000
  blended:     19.77
  ADP:         999.0
────────────────────────────────────────
  VALUE SCORE: 26.69
────────────────────────────────────────
────────────────────────────────────────
  Aaron Judge (The Seamhead Express)
────────────────────────────────────────
  Age:         33  →  age_mult: 0.75x
  FPts:        677.00  (2025 actual)
  fantasy_pts: 561.17  (2026 projected)
  savant_mult: 1.1521
  blended:     607.50
  ADP:         1.9
────────────────────────────────────────
  VALUE SCORE: 524.91
────────────────────────────────────────
────────────────────────────────────────
  Matt Olson (The Seamhead Express)
────────────────────────────────────────
  Age:         31  →  age_mult: 0.88x
  FPts:        506.00 